# Week 4 — Platform Incident Copilot (LangGraph)

Community contribution: **worker → tools → evaluator** loop (same shape as the Sidekick lab), focused on **SRE-style incident triage** without Playwright or cluster credentials.

**Setup:** set `OPENAI_API_KEY` (e.g. repo root `.env`). If imports fail, `%cd` into this folder first.

**Files (≤5 for merge):** `graph.py` (graph + tools) · `app.py` · `requirements.txt` · `week4_platform_incident.ipynb` · `ui_example.png`.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

p = Path("ui_example.png")
if p.is_file():
    display(Image(filename=str(p)))
else:
    print("ui_example.png not found — add the screenshot next to this notebook.")

In [ ]:
from pathlib import Path
import os

from dotenv import load_dotenv

here = Path.cwd()
for d in [here, *here.parents[:8]]:
    env = d / ".env"
    if env.is_file():
        load_dotenv(env, override=False)
        break
load_dotenv(override=False)

if not Path("graph.py").resolve().is_file():
    raise RuntimeError(
        "Start kernel with cwd set to this folder (…/SamuelAdebodun) so graph.py imports cleanly."
    )

from graph import PlatformIncidentCopilot, new_thread_id

print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))
print("cwd:", Path.cwd())

In [ ]:
async def demo():
    bot = PlatformIncidentCopilot()
    tid = new_thread_id()
    out = await bot.arun_turn(
        "Checkout API 503 since 14:10 UTC; nginx ingress shows upstream timeouts; last deploy payments-api 30m ago.",
        "Include hypotheses, ordered checks, and escalation triggers.",
        tid,
    )
    tail = out["messages"][-6:]
    for m in tail:
        txt = str(getattr(m, "content", m))
        preview = txt if len(txt) <= 600 else txt[:600] + "…"
        print(type(m).__name__, "→", preview)

# Uncomment when your API key is available:
await demo()